# Heart Disease — QS-ENN

This notebook is a lightweight dataset-specific entry point. All reusable implementation logic is maintained in `qsenn.py`, ensuring that the same experimental pipeline is applied consistently across datasets.

| Item | Value |
|---|---|
| Dataset key | `heart` |
| Candidate input files | `heart_disease.csv / heart_disease_uci.csv / heart.csv` |
| Target column(s) | `num / target` |
| Excluded variables | `id, dataset` |

The workflow performs dataset preflight checks, fold-isolated preprocessing, fixed-parameter evaluation, ablation analysis, and geometry diagnostics.


## 1. Environment setup


In [ ]:
# Google Colab setup
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# Update these paths for your environment.
PROJECT_DIR = "/content/drive/MyDrive/DRIVES3/qsenn_project/"
RAW_DATA_DIR = "/content/drive/MyDrive/DRIVES3/DATA/"

!pip -q install pennylane imbalanced-learn 2>/dev/null

%run {PROJECT_DIR}qsenn.py
init(data_dir=RAW_DATA_DIR, out_dir=PROJECT_DIR, profile="medium")


## 2. Dataset preflight

Checks file resolution, target selection, excluded variables, class distribution, and potential target-leakage indicators.


In [ ]:
KEY = "heart"

record = load_dataset(KEY)
meta = record["meta"]

print(
    f"n={meta['n']}  features={meta['n_feat']}  classes={meta['n_class']}  "
    f"IR={meta['IR']:.2f}  missing={meta['missing_pct']:.1f}%"
)
print(f"target: {meta['target']}")
print(f"excluded: {meta['dropped'] if meta['dropped'] else 'None'}")
print(f"class distribution: {meta['counts']}")

preflight([KEY])


## 3. Run the dataset experiment

The experiment writes fold-level, descriptor, and ablation CSV files to the configured output directory. Existing results are skipped unless `force=True` is used.


In [ ]:
results, failures = run_all([KEY])
assert not failures, failures


## 4. Mean performance summary


In [ ]:
display(table_one(KEY))


## 5. Three-pillar ablation

Sign convention: **Δ = ablated variant − complete QS-ENN**. A positive Δ means removing or replacing the component increases the metric; a negative Δ means the component contributes positively.


In [ ]:
import pandas as pd

ablation = pd.read_csv(outp(f"ablation_{KEY}.csv"))

display(
    ablation[ablation["metric"] == "F1_1"][
        ["variant", "delta", "N", "z", "p_holm", "significant"]
    ].round(4)
)
display(
    ablation[ablation["metric"] == "Gmean"][
        ["variant", "delta", "N", "z", "p_holm", "significant"]
    ].round(4)
)


## 6. Quantum-geometry descriptors


In [ ]:
import os
import pandas as pd

descriptor_file = outp(f"descriptors_{KEY}.csv")
if os.path.exists(descriptor_file):
    descriptors = pd.read_csv(descriptor_file)
    display(
        descriptors.drop(columns=["key", "rep", "fold"], errors="ignore")
        .groupby("dataset")
        .agg(["mean", "std"])
        .T.round(4)
    )
else:
    print("Geometry descriptors are not available for this dataset.")


## 7. Optional robustness analyses

These analyses are computationally more expensive and can be enabled when needed.


In [ ]:
# nested_df, selected = run_nested_cv(KEY)
# representation_df, representation_pivot = run_representation(KEY)
# finite_shot_df = run_finite_shot(KEY)
